# Pendulum Feature Extraction Speed Comparison

Your goal is to identify the fastest practical feature-extraction architecture for pybasin.

Specifically, measure independently:

1. Feature implementation:
   - Current configurable/loop-heavy extractor.
   - Fused predefined feature calculations.

2. Physical memory layout:
   - TSB: time values for one trajectory are separated in memory.
   - BST: each trajectory’s time values are adjacent.
   - Measure the extraction benefit of physically reordered BST—not merely a permuted view.

3. Execution device:
   - CPU versus GPU extraction.
   - Determine whether GPU batch extraction is worthwhile.

4. Data preparation:
   - CPU↔GPU transfer cost.
   - Physical TSB→BST reorder cost.
   - Whether reordering before or after transfer is faster.

5. Final decision:
   - Quantify gains from fusion alone, BST alone, and both together.
   - Compare those gains on CPU and GPU to determine whether GPU benefits more.

## Reporting plan

Keep extraction and data preparation independent because future solvers may return a different device or physical layout.

1. **Extraction optimization table** — three rows (`BST only`, `fusion only`, `fusion + BST`) with CPU and GPU baseline time, optimized time, speedup, and relative GPU-versus-CPU benefit.
2. **Data preparation table** — extractor-agnostic timings for physical TSB→BST reordering, CPU→GPU and GPU→CPU transfer, and reordering before versus after transfer.
3. **Text conclusions only** — report fusion gain, BST gain, combined gain, whether GPU gains more than CPU, and the fastest physical reorder/transfer route.

Do not add end-to-end rankings or extra plots/tables that repeat these values.

## Before running the benchmark

- **Restart the notebook kernel and run from the first cell.** The JAX allocator environment variables must be set before the first import of JAX, `pybasin`, or any module that imports JAX. Re-running the import cell after JAX is already loaded does not change the active allocator.
- Existing displayed outputs may come from older source code. Treat them as stale until the notebook has been restarted and rerun top to bottom.
- The import cell sets `XLA_PYTHON_CLIENT_PREALLOCATE=false` and `XLA_PYTHON_CLIENT_ALLOCATOR=platform`. This prevents JAX from reserving most GPU memory up front and allows deleted JAX buffers to be returned between integrations. The platform allocator is a benchmark memory-safety choice, not automatically a production-performance recommendation.
- PyTorch and JAX use separate GPU allocators. Always delete the integration result and every PyTorch alias/view that refers to it before starting extraction or the next integration.
- `JaxSolver` may expose a GPU result to PyTorch through DLPack without copying it. In that case, a live PyTorch tensor still owns/references the JAX GPU buffer; deleting only the local JAX objects is insufficient.
- `torch.cuda.empty_cache()` releases only unused cached PyTorch blocks. It cannot release live tensors. Cleanup therefore requires: synchronize, delete all results/views/closures holding tensors, run `gc.collect()`, then call `torch.cuda.empty_cache()`.
- A slice or `permute` is normally a view and can keep the complete backing allocation alive. Materialize only deliberately retained data with a copy, then delete the original solution and all views.
- GPU inputs are transferred once outside extraction timing and reused. GPU outputs are synchronized and deleted after every warm-up and measured repeat. Peak allocated memory is reset and recorded per repeat.

## Experiment notes

- The focused case integrates 50,000 trajectories over `t_span=(0, 2000)` but saves only 990 evaluation points in `t_eval=(1670, 2000)`. In the `JaxSolver` API, `t_steps` is the number of points inside this save region; adaptive integration still begins at time 0.
- Immediately after integration, the saved solution is materialized on CPU and all GPU integration references are deleted before extraction.
- The focused benchmark times extraction from pre-materialized device-resident inputs. CPU-to-GPU input transfer is excluded from extraction timing and measured separately in the preparation-cost table.
- `NORMALIZE` defaults to `False` here so the benchmark stays focused on raw extraction cost. The current estimator default also normalizes; set it to `True` if you want to include that cost too.
- Both compared extractors compute exactly the `DEFAULT_TORCH_FC_PARAMETERS` feature family: `median`, `mean`, `standard_deviation`, `variance`, `root_mean_square`, `maximum`, `absolute_maximum`, `minimum`, `delta`, and `log_delta`.
- The focused extraction table excludes input transfer and layout conversion. All methods include feature assembly, GPU-to-CPU result return, and imputation so their output semantics match.
- Contiguous TSB normally has strides `(B*S, S, 1)`: state is adjacent, but successive time values for one trajectory are `B*S` elements apart. Contiguous BST has strides `(S*T, T, 1)`, making time adjacent.
- `permute(1, 2, 0)` is a metadata-only view. `permute(1, 2, 0).contiguous()` allocates new storage and physically reorders the values.
- CPU↔GPU transfer and physical reordering are reported independently from extraction because future solver output placement/layout may change.

In [1]:
from __future__ import annotations

import gc
import multiprocessing as mp
import os
import subprocess
import sys
import threading
from collections.abc import Callable, Mapping
from pathlib import Path
from time import perf_counter
from typing import Any

# Let deleted JAX integration buffers return memory to the system. These must be
# configured before importing pybasin/JAX modules in a fresh kernel.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

PYBASIN_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "case_studies").exists()
)
PUBLIC_PYBASIN_TARGET = Path("/tmp/public_pybasin_0_2")
TSTORCH_SRC_ROOT = PYBASIN_ROOT / "packages" / "tstorch" / "src"

if not (PUBLIC_PYBASIN_TARGET / "pybasin").exists():
    PUBLIC_PYBASIN_TARGET.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "pybasin==0.2.0",
            "--target",
            str(PUBLIC_PYBASIN_TARGET),
            "--upgrade",
            "--no-deps",
        ],
        check=True,
    )

sys.path.insert(0, str(PUBLIC_PYBASIN_TARGET))
sys.path.append(str(PYBASIN_ROOT))
sys.path.append(str(TSTORCH_SRC_ROOT))

from case_studies.pendulum.setup_pendulum_system import setup_pendulum_system
from pybasin.ts_torch.settings import DEFAULT_TORCH_FC_PARAMETERS
from pybasin.ts_torch.torch_feature_extractor import (
    TorchFeatureExtractor as PublicTorchFeatureExtractor,
)
from pybasin.ts_torch.torch_feature_utilities import impute_extreme

from pybasin.feature_extractors.feature_extractor import FeatureExtractor
from pybasin.feature_extractors.utils import format_feature_name
from pybasin.solution import Solution
from pybasin.solvers import JaxSolver

torch.set_grad_enabled(False)
CUDA_AVAILABLE = torch.cuda.is_available()
print(f"public pybasin 0.2.0: {PUBLIC_PYBASIN_TARGET}")
print(f"CUDA available: {CUDA_AVAILABLE}")

ModuleNotFoundError: No module named 'case_studies'

In [ ]:
TOTAL_T_SPAN = (0.0, 2000.0)
BENCHMARK_T_EVAL = (1670.0, 2000.0)
# JaxSolver.t_steps means evaluation points inside t_eval, not across t_span.
# 990 points over 330 time units preserves approximately 3 saved points/unit.
SAVE_REGION_EVALUATION_POINTS = 990
TARGET_BATCH_SIZE = 50_000
REPEATS = 15
WARMUP_RUNS = 3
NORMALIZE = False

DEFAULT_FEATURE_ORDER = list(DEFAULT_TORCH_FC_PARAMETERS.keys())

In [ ]:
class FusedDefaultFeatureExtractor(FeatureExtractor):
    """Benchmark-only extractor that computes exactly DEFAULT_TORCH_FC_PARAMETERS."""

    def __init__(self, time_steady: float = 0.0, device: str = "cpu"):
        super().__init__(time_steady=time_steady)
        if device == "gpu" and not CUDA_AVAILABLE:
            raise RuntimeError("CUDA is not available but device='gpu' was requested")
        self.device = device
        self._feature_names_cache: list[str] | None = None

    def _build_feature_names(self, n_states: int) -> list[str]:
        return [
            format_feature_name(name, state_index=state_idx)
            for state_idx in range(n_states)
            for name in DEFAULT_FEATURE_ORDER
        ]

    @property
    def feature_names(self) -> list[str]:
        if self._feature_names_cache is None:
            raise RuntimeError("Feature names not available. Call extract_features first.")
        return self._feature_names_cache

    def extract_features(self, solution: Solution) -> torch.Tensor:
        y = self.filter_time(solution)
        if self.device == "gpu":
            y = y.to("cuda")
        else:
            y = y.cpu()

        _, _, n_states = y.shape
        self._feature_names_cache = self._build_feature_names(n_states)
        features = flatten_default_feature_results(compute_default_feature_results_tsb(y))
        if features.is_cuda:
            features = features.cpu()
        return impute_extreme(features)


def compute_default_feature_results_tsb(y_tsb: torch.Tensor) -> dict[str, torch.Tensor]:
    mean_x = y_tsb.mean(dim=0)
    max_x = y_tsb.max(dim=0).values
    delta_x = (max_x - mean_x).abs()

    return {
        "median": y_tsb.median(dim=0).values,
        "mean": mean_x,
        "standard_deviation": y_tsb.std(dim=0, correction=0),
        "variance": y_tsb.var(dim=0, correction=0),
        "root_mean_square": torch.sqrt((y_tsb**2).mean(dim=0)),
        "maximum": max_x,
        "absolute_maximum": y_tsb.abs().max(dim=0).values,
        "minimum": y_tsb.min(dim=0).values,
        "delta": delta_x,
        "log_delta": torch.log(delta_x + 1e-10),
    }


def compute_default_feature_results_bst(y_bst: torch.Tensor) -> dict[str, torch.Tensor]:
    median_x = y_bst.median(dim=-1).values
    mean_x = y_bst.mean(dim=-1)
    std_x = y_bst.std(dim=-1, correction=0)
    var_x = y_bst.var(dim=-1, correction=0)
    rms_x = torch.sqrt((y_bst**2).mean(dim=-1))
    max_x = y_bst.max(dim=-1).values
    abs_max_x = y_bst.abs().max(dim=-1).values
    min_x = y_bst.min(dim=-1).values

    delta_x = (max_x - mean_x).abs()
    log_delta_x = torch.log(delta_x + 1e-10)

    return {
        "median": median_x,
        "mean": mean_x,
        "standard_deviation": std_x,
        "variance": var_x,
        "root_mean_square": rms_x,
        "maximum": max_x,
        "absolute_maximum": abs_max_x,
        "minimum": min_x,
        "delta": delta_x,
        "log_delta": log_delta_x,
    }


def flatten_default_feature_results(results: dict[str, torch.Tensor]) -> torch.Tensor:
    feature_tensors = [results[name] for name in DEFAULT_FEATURE_ORDER]
    return torch.stack(feature_tensors, dim=2).reshape(feature_tensors[0].shape[0], -1)


def assert_feature_matrices_close(
    reference: torch.Tensor,
    candidate: torch.Tensor,
    label: str,
    *,
    source_tsb: torch.Tensor | None = None,
) -> None:
    if reference.shape != candidate.shape:
        raise AssertionError(
            f"{label}: shape mismatch {tuple(reference.shape)} != {tuple(candidate.shape)}"
        )

    feature_count = len(DEFAULT_FEATURE_ORDER)
    reference_view = reference.reshape(reference.shape[0], -1, feature_count)
    candidate_view = candidate.reshape(candidate.shape[0], -1, feature_count)

    # TSB and BST reductions traverse identical float32 values with different
    # reduction trees. Bound the expected summation error using eps * T * max|x|.
    if source_tsb is not None:
        n_time = source_tsb.shape[0]
        max_abs_source = torch.maximum(
            source_tsb.amax(dim=0).abs(),
            source_tsb.amin(dim=0).abs(),
        ).cpu()
        reduction_atol = 8.0 * torch.finfo(source_tsb.dtype).eps * n_time * max_abs_source
        reduction_atol = reduction_atol.clamp_min(1e-6)
    else:
        reduction_atol = torch.full_like(reference_view[:, :, 0], 1e-5)

    failures: list[str] = []
    for feature_idx, feature_name in enumerate(DEFAULT_FEATURE_ORDER):
        reference_feature = reference_view[:, :, feature_idx]
        candidate_feature = candidate_view[:, :, feature_idx]
        feature_diff = (reference_feature - candidate_feature).abs()

        if feature_name in {"median", "maximum", "absolute_maximum", "minimum"}:
            valid = torch.equal(reference_feature, candidate_feature)
        elif feature_name in {"mean", "delta"}:
            valid = bool(torch.all(feature_diff <= reduction_atol))
        elif feature_name == "log_delta":
            # log is ill-conditioned near delta=0. Convert the accepted delta
            # error bound into the corresponding local log-domain bound.
            delta_idx = DEFAULT_FEATURE_ORDER.index("delta")
            reference_delta = reference_view[:, :, delta_idx].abs()
            candidate_delta = candidate_view[:, :, delta_idx].abs()
            min_delta = torch.minimum(reference_delta, candidate_delta)
            log_atol = torch.log1p(reduction_atol / (min_delta + 1e-10)) + 1e-6
            valid = bool(torch.all(feature_diff <= log_atol))
        else:
            valid = torch.allclose(reference_feature, candidate_feature, atol=1e-5, rtol=1e-4)

        if not valid:
            failures.append(f"{feature_name}={feature_diff.max().item():.3e}")

    if failures:
        raise AssertionError(f"{label}: feature parity failed: {', '.join(failures)}")


def make_extractors() -> list[tuple[str, str, FeatureExtractor]]:
    extractors: list[tuple[str, str, FeatureExtractor]] = [
        (
            "torch_default",
            "cpu",
            PublicTorchFeatureExtractor(
                time_steady=0.0,
                features=DEFAULT_TORCH_FC_PARAMETERS,
                normalize=NORMALIZE,
                device="cpu",
            ),
        ),
        (
            "fused_default",
            "cpu",
            FusedDefaultFeatureExtractor(time_steady=0.0, device="cpu"),
        ),
    ]

    if CUDA_AVAILABLE:
        extractors.extend([
            (
                "torch_default",
                "gpu",
                PublicTorchFeatureExtractor(
                    time_steady=0.0,
                    features=DEFAULT_TORCH_FC_PARAMETERS,
                    normalize=NORMALIZE,
                    device="gpu",
                ),
            ),
            (
                "fused_default",
                "gpu",
                FusedDefaultFeatureExtractor(time_steady=0.0, device="gpu"),
            ),
        ])

    return extractors


def compute_default_feature_bst(x: torch.Tensor, feature_name: str) -> torch.Tensor:
    """Compute one configurable default feature over stride-1 BST time data."""
    if feature_name == "median":
        return x.median(dim=-1).values
    if feature_name == "mean":
        return x.mean(dim=-1)
    if feature_name == "standard_deviation":
        return x.std(dim=-1, correction=0)
    if feature_name == "variance":
        return x.var(dim=-1, correction=0)
    if feature_name == "root_mean_square":
        return torch.sqrt((x**2).mean(dim=-1))
    if feature_name == "maximum":
        return x.max(dim=-1).values
    if feature_name == "absolute_maximum":
        return x.abs().max(dim=-1).values
    if feature_name == "minimum":
        return x.min(dim=-1).values
    if feature_name == "delta":
        return (x.max(dim=-1).values - x.mean(dim=-1)).abs()
    if feature_name == "log_delta":
        delta_x = (x.max(dim=-1).values - x.mean(dim=-1)).abs()
        return torch.log(delta_x + 1e-10)
    raise KeyError(f"BST benchmark kernel is not defined for {feature_name!r}")


def _extract_features_bst_chunk_worker(
    args: tuple[Any, list[str]],
) -> dict[str, Any]:
    chunk_np, feature_names = args
    chunk = torch.from_numpy(chunk_np)
    torch.set_num_threads(1)
    return {
        feature_name: compute_default_feature_bst(chunk, feature_name).numpy()
        for feature_name in feature_names
    }


def extract_features_bst_parallel(
    x_bst: torch.Tensor,
    feature_names: list[str],
    n_workers: int,
) -> dict[str, torch.Tensor]:
    """Mirror the production CPU chunking path for contiguous (B, S, T) input."""
    x_np = x_bst.numpy()
    n_batches = x_np.shape[0]
    n_workers = max(1, min(n_workers, n_batches))
    chunk_size = max(1, n_batches // n_workers)
    worker_args = [
        (x_np[start : min(start + chunk_size, n_batches)].copy(), feature_names)
        for start in range(0, n_batches, chunk_size)
    ]
    with mp.Pool(processes=n_workers) as pool:
        chunk_results = pool.map(_extract_features_bst_chunk_worker, worker_args)
    torch.set_num_threads(os.cpu_count() or 1)
    return {
        feature_name: torch.from_numpy(
            np.concatenate([chunk_result[feature_name] for chunk_result in chunk_results], axis=0)
        )
        for feature_name in feature_names
    }


class TorchFeatureExtractorBST:
    """Notebook-local BST counterpart of TorchFeatureExtractor for the default kernels."""

    def __init__(
        self,
        features: Mapping[str, Any] | None = DEFAULT_TORCH_FC_PARAMETERS,
        features_per_state: dict[int, Mapping[str, Any] | None] | None = None,
        normalize: bool = True,
        device: str = "cpu",
        n_jobs: int | None = None,
    ):
        if device == "gpu" and not CUDA_AVAILABLE:
            raise RuntimeError("CUDA is not available but device='gpu' was requested")
        self.features = features
        self.features_per_state = features_per_state or {}
        self.normalize = normalize
        self.device = device
        self.n_jobs = n_jobs if n_jobs is not None else (os.cpu_count() or 1)
        self._fit_lock = threading.Lock()
        self._is_fitted = False
        self._feature_mean: torch.Tensor | None = None
        self._feature_std: torch.Tensor | None = None
        self._resulting_features_config: dict[int, Mapping[str, Any]] | None = None
        self._num_states: int | None = None
        self._extracted_feature_names: list[str] | None = None

    def _configure_resulting_features_config(self, num_states: int) -> None:
        if self._resulting_features_config is not None and self._num_states == num_states:
            return
        self._num_states = num_states
        self._resulting_features_config = {}
        for state_idx in range(num_states):
            if state_idx in self.features_per_state:
                fc_params = self.features_per_state[state_idx]
                if fc_params is not None:
                    self._resulting_features_config[state_idx] = fc_params
            elif self.features is not None:
                self._resulting_features_config[state_idx] = self.features

    def _is_uniform_config(self) -> bool:
        if not self._resulting_features_config:
            return True
        configs = list(self._resulting_features_config.values())
        return len(configs) <= 1 or all(config is configs[0] for config in configs[1:])

    @staticmethod
    def _validate_feature_config(fc_params: Mapping[str, Any]) -> list[str]:
        unsupported = [
            name
            for name, params in fc_params.items()
            if name not in DEFAULT_FEATURE_ORDER or params is not None
        ]
        if unsupported:
            raise ValueError(
                f"BST benchmark supports only parameter-free default features: {unsupported}"
            )
        return list(fc_params.keys())

    def _extract_all_states(
        self,
        y_bst: torch.Tensor,
        fc_params: Mapping[str, Any],
    ) -> dict[str, torch.Tensor]:
        feature_names = self._validate_feature_config(fc_params)
        if self.device == "gpu":
            return {name: compute_default_feature_bst(y_bst, name) for name in feature_names}
        return extract_features_bst_parallel(y_bst, feature_names, self.n_jobs)

    def _extract_for_state(
        self,
        y_state_bst: torch.Tensor,
        fc_params: Mapping[str, Any],
    ) -> dict[str, torch.Tensor]:
        return {
            name: value[:, 0] if value.ndim == 2 else value
            for name, value in self._extract_all_states(y_state_bst, fc_params).items()
        }

    def extract_features_preordered(self, y_bst: torch.Tensor) -> torch.Tensor:
        if y_bst.ndim != 3 or not y_bst.is_contiguous():
            raise ValueError("Expected contiguous BST input with shape (batch, state, time)")
        num_states = y_bst.shape[1]
        self._configure_resulting_features_config(num_states)
        assert self._resulting_features_config is not None
        if not self._resulting_features_config:
            return torch.zeros((y_bst.shape[0], 0), dtype=y_bst.dtype)

        all_features: list[torch.Tensor] = []
        all_feature_names: list[str] = []
        if self._is_uniform_config():
            fc_params = next(iter(self._resulting_features_config.values()))
            state_indices = list(self._resulting_features_config.keys())
            if state_indices == list(range(num_states)):
                # Avoid advanced indexing: y_bst[:, [0, 1, ...], :] silently
                # allocates a full copy and hides the benefit of BST memory order.
                y_selected = y_bst
            else:
                y_selected = y_bst.index_select(
                    1,
                    torch.tensor(state_indices, device=y_bst.device),
                )
            results = self._extract_all_states(y_selected, fc_params)
            for state_pos, state_idx in enumerate(state_indices):
                for feature_name in fc_params:
                    all_features.append(results[feature_name][:, state_pos])
                    all_feature_names.append(
                        format_feature_name(feature_name, state_index=state_idx)
                    )
        else:
            for state_idx, fc_params in self._resulting_features_config.items():
                results = self._extract_for_state(y_bst[:, state_idx : state_idx + 1, :], fc_params)
                for feature_name in fc_params:
                    all_features.append(results[feature_name])
                    all_feature_names.append(
                        format_feature_name(feature_name, state_index=state_idx)
                    )

        self._extracted_feature_names = all_feature_names
        features = torch.stack(all_features, dim=1)
        if self.device == "gpu":
            features = features.cpu()
        features = impute_extreme(features)
        if self.normalize:
            with self._fit_lock:
                if not self._is_fitted:
                    self._feature_mean = features.mean(dim=0, keepdim=True)
                    self._feature_std = features.std(dim=0, keepdim=True)
                    self._feature_std = torch.where(
                        self._feature_std == 0,
                        torch.ones_like(self._feature_std),
                        self._feature_std,
                    )
                    self._is_fitted = True
                assert self._feature_mean is not None and self._feature_std is not None
                features = (features - self._feature_mean) / self._feature_std
        return features

    def reset_scaler(self) -> None:
        with self._fit_lock:
            self._feature_mean = None
            self._feature_std = None
            self._is_fitted = False

    @property
    def feature_names(self) -> list[str]:
        if self._extracted_feature_names is None:
            raise RuntimeError(
                "Feature names not available. Call extract_features_preordered first."
            )
        return self._extracted_feature_names


def extract_fused_default_tsb(y_tsb: torch.Tensor) -> torch.Tensor:
    features = flatten_default_feature_results(compute_default_feature_results_tsb(y_tsb))
    return impute_extreme(features.cpu() if features.is_cuda else features)


def extract_fused_default_bst(y_bst: torch.Tensor) -> torch.Tensor:
    features = flatten_default_feature_results(compute_default_feature_results_bst(y_bst))
    return impute_extreme(features.cpu() if features.is_cuda else features)


def extract_fused_default_bst_device_resident(y_bst: torch.Tensor) -> torch.Tensor:
    features = flatten_default_feature_results(compute_default_feature_results_bst(y_bst))
    return impute_extreme(features)

In [ ]:
setup = setup_pendulum_system()
ode_system = setup["ode_system"]
sampler = setup["sampler"]
base_solver = setup["solver"]
solver_device = "gpu" if CUDA_AVAILABLE else "cpu"


def synchronize(device: str) -> None:
    if device == "gpu" and CUDA_AVAILABLE:
        torch.cuda.synchronize()


def release_gpu_memory() -> None:
    if CUDA_AVAILABLE:
        torch.cuda.synchronize()
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()


def materialize_tensor_on_device(x: torch.Tensor, device: str) -> torch.Tensor:
    """Move a tensor and sever any view retaining a larger backing allocation."""
    target = torch.device(device)
    if x.device == target:
        return x.detach().clone().contiguous()
    return x.detach().to(target).contiguous()


def clone_solution_to_device(solution: Solution, device: str) -> Solution:
    return Solution(
        materialize_tensor_on_device(solution.initial_condition, device),
        materialize_tensor_on_device(solution.time, device),
        materialize_tensor_on_device(solution.y, device),
    )


def integrate_batch(batch_size: int) -> Solution:
    y0 = sampler.sample(batch_size)
    solver = JaxSolver(
        t_span=TOTAL_T_SPAN,
        t_steps=SAVE_REGION_EVALUATION_POINTS,
        t_eval=BENCHMARK_T_EVAL,
        device=solver_device,
        rtol=base_solver.rtol,
        atol=base_solver.atol,
        cache_dir=None,
    )
    time_arr, y_arr = solver.integrate(ode_system, y0)
    assert time_arr.shape == (SAVE_REGION_EVALUATION_POINTS,)
    assert y_arr.shape[0] == SAVE_REGION_EVALUATION_POINTS
    assert abs(float(time_arr[0]) - BENCHMARK_T_EVAL[0]) < 1e-4
    assert abs(float(time_arr[-1]) - BENCHMARK_T_EVAL[1]) < 1e-4
    return Solution(y0, time_arr, y_arr)

In [ ]:
# Integrate the single focused case and verify the physical layout conversion.
release_gpu_memory()
focused_solution = integrate_batch(TARGET_BATCH_SIZE)
focused_solver_y = focused_solution.y
focused_solver_output_layout = {
    "shape": tuple(focused_solver_y.shape),
    "stride": focused_solver_y.stride(),
    "is_contiguous": focused_solver_y.is_contiguous(),
    "dtype": str(focused_solver_y.dtype),
    "device": str(focused_solver_y.device),
    "data_ptr": focused_solver_y.data_ptr(),
    "storage_offset": focused_solver_y.storage_offset(),
    "nbytes": focused_solver_y.untyped_storage().nbytes(),
}
focused_solution_cpu = clone_solution_to_device(focused_solution, "cpu")
del focused_solver_y, focused_solution
release_gpu_memory()

solver_y_tsb = focused_solution_cpu.y
y_bst_view = solver_y_tsb.permute(1, 2, 0)
y_bst_contig = y_bst_view.contiguous()
y_tsb_view_from_bst = y_bst_contig.permute(2, 0, 1)

assert solver_y_tsb.shape[0] == SAVE_REGION_EVALUATION_POINTS
assert solver_y_tsb.shape[1] == TARGET_BATCH_SIZE
assert y_bst_view.data_ptr() == solver_y_tsb.data_ptr()
assert y_bst_contig.is_contiguous()
assert y_bst_contig.stride()[-1] == 1
assert y_bst_contig.data_ptr() != solver_y_tsb.data_ptr()
assert torch.equal(solver_y_tsb, y_tsb_view_from_bst)

del solver_y_tsb, y_bst_view, y_bst_contig, y_tsb_view_from_bst
gc.collect()

In [ ]:
# Focused 2x2 extraction benchmark plus CPU-to-GPU and layout preparation costs


def benchmark_callable(
    label: str,
    device: str,
    fn: Callable[[], Any],
    *,
    calls_per_repeat: int = 1,
) -> dict[str, float | str]:
    if device == "gpu":
        release_gpu_memory()
    for _ in range(WARMUP_RUNS):
        warmup_result = fn()
        synchronize(device)
        del warmup_result
        if device == "gpu":
            release_gpu_memory()

    times_ms: list[float] = []
    peak_allocated_gb: list[float] = []
    for _ in range(REPEATS):
        if device == "gpu":
            release_gpu_memory()
            torch.cuda.reset_peak_memory_stats()
        synchronize(device)
        start = perf_counter()
        for _ in range(calls_per_repeat):
            result = fn()
        synchronize(device)
        times_ms.append((perf_counter() - start) * 1000.0 / calls_per_repeat)
        if device == "gpu":
            peak_allocated_gb.append(torch.cuda.max_memory_allocated() / 1e9)
        del result
        if device == "gpu":
            release_gpu_memory()

    return {
        "operation": label,
        "device": device,
        "median_ms": float(pd.Series(times_ms).median()),
        "min_ms": float(min(times_ms)),
        "mean_ms": float(sum(times_ms) / len(times_ms)),
        "max_ms": float(max(times_ms)),
        "peak_allocated_gb": float(max(peak_allocated_gb)) if peak_allocated_gb else 0.0,
    }


def make_resident_solution(reference: Solution, y: torch.Tensor) -> Solution:
    return Solution(
        reference.initial_condition.to(y.device),
        reference.time.to(y.device),
        y,
    )


def payload_bandwidth_gb_s(payload_bytes: int, elapsed_ms: float) -> float:
    return payload_bytes / (elapsed_ms * 1e6)


solution_50k_steady = focused_solution_cpu
assert solution_50k_steady.y.shape == (
    SAVE_REGION_EVALUATION_POINTS,
    TARGET_BATCH_SIZE,
    solution_50k_steady.initial_condition.shape[1],
)

# These one-time materializations are deliberately outside extraction timing.
solver_y_tsb_cpu = solution_50k_steady.y
y_tsb_cpu = solver_y_tsb_cpu.contiguous()
y_bst_cpu = y_tsb_cpu.permute(1, 2, 0).contiguous()
solution_tsb_cpu = make_resident_solution(solution_50k_steady, y_tsb_cpu)

assert y_tsb_cpu.is_contiguous() and y_bst_cpu.is_contiguous()
assert y_tsb_cpu.shape == (SAVE_REGION_EVALUATION_POINTS, TARGET_BATCH_SIZE, y_bst_cpu.shape[1])
assert y_bst_cpu.shape == (TARGET_BATCH_SIZE, y_tsb_cpu.shape[2], SAVE_REGION_EVALUATION_POINTS)

y_tsb_gpu = y_tsb_cpu.to("cuda") if CUDA_AVAILABLE else None
y_bst_gpu = y_bst_cpu.to("cuda") if CUDA_AVAILABLE else None
solution_tsb_gpu = (
    make_resident_solution(solution_50k_steady, y_tsb_gpu) if y_tsb_gpu is not None else None
)

configurable_tsb_cpu = PublicTorchFeatureExtractor(
    time_steady=0.0,
    features=DEFAULT_TORCH_FC_PARAMETERS,
    normalize=NORMALIZE,
    device="cpu",
)
configurable_bst_cpu = TorchFeatureExtractorBST(
    features=DEFAULT_TORCH_FC_PARAMETERS,
    normalize=NORMALIZE,
    device="cpu",
)
configurable_tsb_gpu = (
    PublicTorchFeatureExtractor(
        time_steady=0.0,
        features=DEFAULT_TORCH_FC_PARAMETERS,
        normalize=NORMALIZE,
        device="gpu",
    )
    if CUDA_AVAILABLE
    else None
)
configurable_bst_gpu = (
    TorchFeatureExtractorBST(
        features=DEFAULT_TORCH_FC_PARAMETERS,
        normalize=NORMALIZE,
        device="gpu",
    )
    if CUDA_AVAILABLE
    else None
)

extraction_calls: dict[tuple[str, str, str], Callable[[], torch.Tensor]] = {
    ("configurable", "TSB", "cpu"): lambda: configurable_tsb_cpu.extract_features(solution_tsb_cpu),
    ("configurable", "BST", "cpu"): lambda: configurable_bst_cpu.extract_features_preordered(
        y_bst_cpu
    ),
    ("fused", "TSB", "cpu"): lambda: extract_fused_default_tsb(y_tsb_cpu),
    ("fused", "BST", "cpu"): lambda: extract_fused_default_bst(y_bst_cpu),
}
if CUDA_AVAILABLE:
    assert solution_tsb_gpu is not None and y_tsb_gpu is not None and y_bst_gpu is not None
    assert configurable_tsb_gpu is not None and configurable_bst_gpu is not None
    extraction_calls.update({
        ("configurable", "TSB", "gpu"): lambda: configurable_tsb_gpu.extract_features(
            solution_tsb_gpu
        ),
        ("configurable", "BST", "gpu"): lambda: configurable_bst_gpu.extract_features_preordered(
            y_bst_gpu
        ),
        ("fused", "TSB", "gpu"): lambda: extract_fused_default_tsb(y_tsb_gpu),
        ("fused", "BST", "gpu"): lambda: extract_fused_default_bst(y_bst_gpu),
    })


def cpu_for_validation(features: torch.Tensor) -> torch.Tensor:
    return features.detach().cpu() if features.is_cuda else features.detach()


# Validate values and feature ordering before timing. Device movement here is
# validation-only and not part of the measured extraction calls.
reference_features = cpu_for_validation(extraction_calls[("configurable", "TSB", "cpu")]())
for key, fn in extraction_calls.items():
    candidate_features = fn()
    assert_feature_matrices_close(
        reference_features,
        cpu_for_validation(candidate_features),
        " / ".join(key),
        source_tsb=y_tsb_cpu,
    )
    del candidate_features
assert configurable_tsb_cpu.feature_names == configurable_bst_cpu.feature_names
if CUDA_AVAILABLE:
    assert configurable_tsb_gpu is not None and configurable_bst_gpu is not None
    assert configurable_tsb_gpu.feature_names == configurable_bst_gpu.feature_names

extraction_rows: list[dict[str, float | str]] = []
for (implementation, layout, device), fn in extraction_calls.items():
    timing = benchmark_callable(f"{implementation}_{layout.lower()}", device, fn)
    extraction_rows.append({
        "implementation": implementation,
        "layout": layout,
        "device": device,
        **timing,
    })

extraction_long_df = pd.DataFrame(extraction_rows)


def extraction_ms(implementation: str, layout: str, device: str) -> float:
    row = extraction_long_df[
        (extraction_long_df["implementation"] == implementation)
        & (extraction_long_df["layout"] == layout)
        & (extraction_long_df["device"] == device)
    ]
    return float(row.iloc[0]["median_ms"]) if not row.empty else float("nan")


# Preparation costs are extractor-agnostic and begin from one physical TSB tensor.
payload_bytes = y_tsb_cpu.numel() * y_tsb_cpu.element_size()
payload_gb = payload_bytes / 1e9
preparation_rows: list[dict[str, float | str]] = []


def record_preparation(
    comparison: str,
    route_id: str,
    route: str,
    device: str,
    fn: Callable[[], torch.Tensor],
    expected_shape: tuple[int, ...],
) -> None:
    validation_result = fn()
    synchronize(device)
    assert tuple(validation_result.shape) == expected_shape
    assert validation_result.is_contiguous()
    del validation_result
    if device == "gpu":
        release_gpu_memory()

    timing = benchmark_callable(route_id, device, fn)
    median_ms = float(timing["median_ms"])
    preparation_rows.append({
        "comparison": comparison,
        "route": route,
        "payload_gb": payload_gb,
        "median_ms": median_ms,
        "effective_payload_gb_s": payload_bandwidth_gb_s(payload_bytes, median_ms),
    })


record_preparation(
    "Physical TSB → BST reorder",
    "reorder_cpu_tsb_to_bst",
    "Reorder on CPU",
    "cpu",
    lambda: y_tsb_cpu.permute(1, 2, 0).contiguous(),
    tuple(y_bst_cpu.shape),
)

if CUDA_AVAILABLE:
    assert y_tsb_gpu is not None and y_bst_gpu is not None
    preparation_routes: list[
        tuple[str, str, str, str, Callable[[], torch.Tensor], tuple[int, ...]]
    ] = [
        (
            "Physical TSB → BST reorder",
            "reorder_resident_gpu_tsb",
            "Reorder on GPU",
            "gpu",
            lambda: y_tsb_gpu.permute(1, 2, 0).contiguous(),
            tuple(y_bst_gpu.shape),
        ),
        (
            "CPU → GPU transfer, keep TSB",
            "transfer_tsb_to_gpu",
            "Transfer TSB only",
            "gpu",
            lambda: y_tsb_cpu.to("cuda"),
            tuple(y_tsb_gpu.shape),
        ),
        (
            "CPU TSB → GPU BST",
            "reorder_cpu_then_transfer_bst",
            "Reorder on CPU, then transfer",
            "gpu",
            lambda: y_tsb_cpu.permute(1, 2, 0).contiguous().to("cuda"),
            tuple(y_bst_gpu.shape),
        ),
        (
            "CPU TSB → GPU BST",
            "transfer_tsb_then_reorder_gpu",
            "Transfer TSB, then reorder on GPU",
            "gpu",
            lambda: y_tsb_cpu.to("cuda").permute(1, 2, 0).contiguous(),
            tuple(y_bst_gpu.shape),
        ),
        (
            "CPU TSB → GPU BST",
            "transfer_bst_shaped_view",
            "Transfer directly from a BST-shaped CPU view",
            "gpu",
            lambda: y_tsb_cpu.permute(1, 2, 0).to("cuda").contiguous(),
            tuple(y_bst_gpu.shape),
        ),
        (
            "GPU → CPU transfer, keep TSB",
            "transfer_gpu_tsb_to_cpu",
            "Transfer TSB only",
            "gpu",
            lambda: y_tsb_gpu.cpu(),
            tuple(y_tsb_cpu.shape),
        ),
        (
            "GPU TSB → CPU BST",
            "reorder_gpu_then_transfer_bst_cpu",
            "Reorder on GPU, then transfer",
            "gpu",
            lambda: y_tsb_gpu.permute(1, 2, 0).contiguous().cpu(),
            tuple(y_bst_cpu.shape),
        ),
        (
            "GPU TSB → CPU BST",
            "transfer_gpu_tsb_then_reorder_cpu",
            "Transfer TSB, then reorder on CPU",
            "gpu",
            lambda: y_tsb_gpu.cpu().permute(1, 2, 0).contiguous(),
            tuple(y_bst_cpu.shape),
        ),
    ]
    for route_args in preparation_routes:
        record_preparation(*route_args)

preparation_df = pd.DataFrame(preparation_rows)
print("Data preparation costs (one logical trajectory payload per row):")
display(
    preparation_df[
        [
            "comparison",
            "route",
            "payload_gb",
            "median_ms",
            "effective_payload_gb_s",
        ]
    ].round(3)
)

# Release all large focused benchmark tensors and callable closures after reporting.
del extraction_calls, reference_features
del solver_y_tsb_cpu, y_tsb_cpu, solution_tsb_cpu
del y_tsb_gpu, solution_tsb_gpu
del configurable_tsb_cpu, configurable_bst_cpu
del configurable_tsb_gpu, configurable_bst_gpu
del solution_50k_steady, focused_solution_cpu
if CUDA_AVAILABLE:
    del preparation_routes
release_gpu_memory()

In [ ]:
# Decision report: extraction gains only; preparation remains independent above.
cpu_current_tsb_ms = extraction_ms("configurable", "TSB", "cpu")
gpu_current_tsb_ms = extraction_ms("configurable", "TSB", "gpu") if CUDA_AVAILABLE else np.nan
current_gpu_vs_cpu_speedup = cpu_current_tsb_ms / gpu_current_tsb_ms if CUDA_AVAILABLE else np.nan

optimization_cases = [
    ("BST layout only", "configurable", "BST"),
    ("Fused calculations only", "fused", "TSB"),
    ("Fused calculations + BST layout", "fused", "BST"),
]

optimization_rows: list[dict[str, float | str]] = []
for optimization, implementation, layout in optimization_cases:
    cpu_optimized_ms = extraction_ms(implementation, layout, "cpu")
    speedup_vs_current_cpu_tsb = cpu_current_tsb_ms / cpu_optimized_ms
    gpu_optimized_ms = extraction_ms(implementation, layout, "gpu") if CUDA_AVAILABLE else np.nan
    speedup_vs_current_gpu_tsb = gpu_current_tsb_ms / gpu_optimized_ms if CUDA_AVAILABLE else np.nan
    optimized_gpu_vs_cpu_speedup = cpu_optimized_ms / gpu_optimized_ms if CUDA_AVAILABLE else np.nan
    optimization_rows.append({
        "optimization": optimization,
        "cpu_current_tsb_ms": cpu_current_tsb_ms,
        "cpu_optimized_ms": cpu_optimized_ms,
        "speedup_vs_current_cpu_tsb": speedup_vs_current_cpu_tsb,
        "gpu_current_tsb_ms": gpu_current_tsb_ms,
        "gpu_optimized_ms": gpu_optimized_ms,
        "speedup_vs_current_gpu_tsb": speedup_vs_current_gpu_tsb,
        "optimized_gpu_vs_cpu_speedup": optimized_gpu_vs_cpu_speedup,
        "gpu_optimization_gain_vs_cpu": (
            speedup_vs_current_gpu_tsb / speedup_vs_current_cpu_tsb if CUDA_AVAILABLE else np.nan
        ),
    })

optimization_summary_df = pd.DataFrame(optimization_rows)
print("Extraction optimizations relative to the current configurable TSB extractor:")
display(optimization_summary_df.round(3))

if CUDA_AVAILABLE:
    print(f"Current extractor, GPU versus CPU: {current_gpu_vs_cpu_speedup:.2f}x faster.")

for row in optimization_rows:
    if CUDA_AVAILABLE:
        print(
            f"{row['optimization']}: CPU {row['speedup_vs_current_cpu_tsb']:.2f}x; "
            f"GPU {row['speedup_vs_current_gpu_tsb']:.2f}x; "
            f"optimized GPU versus optimized CPU "
            f"{row['optimized_gpu_vs_cpu_speedup']:.2f}x; "
            f"GPU receives {row['gpu_optimization_gain_vs_cpu']:.2f}x "
            f"the optimization gain of CPU."
        )
    else:
        print(f"{row['optimization']}: CPU {row['speedup_vs_current_cpu_tsb']:.2f}x.")

if CUDA_AVAILABLE:
    fastest_cpu_to_gpu = (
        preparation_df[preparation_df["comparison"] == "CPU TSB → GPU BST"]
        .nsmallest(1, "median_ms")
        .iloc[0]
    )
    fastest_gpu_to_cpu = (
        preparation_df[preparation_df["comparison"] == "GPU TSB → CPU BST"]
        .nsmallest(1, "median_ms")
        .iloc[0]
    )
    print(
        f"Fastest CPU TSB → GPU BST route: {fastest_cpu_to_gpu['route']} "
        f"({fastest_cpu_to_gpu['median_ms']:.2f} ms)."
    )
    print(
        f"Fastest GPU TSB → CPU BST route: {fastest_gpu_to_cpu['route']} "
        f"({fastest_gpu_to_cpu['median_ms']:.2f} ms)."
    )

## API design comparison: FunctionKernel vs ModuleNN vs BaseClass

The layout and fusion comparison above (see the extraction optimization table
and decision report) established physical BST layout and fused feature
calculations as the faster combination, on both CPU and GPU. From this point
forward, every extractor variant benchmarked below uses BST layout and one
fused, `torch.compile`d kernel per configuration - that choice is held fixed
as the winner from the previous phase, not re-benchmarked here. What varies
between the three variants below is a different, independent axis: the
Python-level API/container shape used to express that fused kernel.

- **`FunctionKernelFeatureExtractor`** - bare closures/functions in a list,
  concatenated with `torch.cat` and compiled as one `torch.compile` call.
- **`ModuleNNFeatureExtractor`** - one `nn.Module` per feature, held in an
  `nn.ModuleList`, composed with the same `torch.cat` + `torch.compile`
  pattern.
- **`BaseClassFeatureExtractor`** - one plain Python class per feature (no
  `nn.Module` inheritance - no hooks, no training-mode dispatch, no
  buffer/parameter registration), composed the same way.

This section assumes the trajectory representation is already a physical BST
tensor `(batch, state, time)` on the device where extraction will run.

The extractor does not reorder data, move data between CPU and GPU, or
normalize results. Its only public operation is `extract_features(y_bst)`.
Configuration dictionaries are compiled once into, per requested feature, the
list of states that actually asked for it; each feature's reduction runs only
over those states (sliced out first via `index_select`, skipped entirely when
every state wants it) instead of computing over every state and discarding the
unwanted results afterward.


In [ ]:
# One independent function per feature - the actual per-feature calculation, not a
# branch inside a bigger function. Each takes the full (B, S, T) tensor and reduces
# over time, returning (B, S). The Configurable extractor below decides *which* of
# these to call based on the requested config, then builds one composed kernel out
# of exactly that subset - it does not hardcode "compute all 10, discard the rest".
FEATURE_FUNCS_BST: dict[str, Callable[[torch.Tensor], torch.Tensor]] = {
    "median": lambda x: x.median(dim=-1).values,
    "mean": lambda x: x.mean(dim=-1),
    "standard_deviation": lambda x: x.std(dim=-1, correction=0),
    "variance": lambda x: x.var(dim=-1, correction=0),
    "root_mean_square": lambda x: torch.sqrt((x**2).mean(dim=-1)),
    "maximum": lambda x: x.max(dim=-1).values,
    "absolute_maximum": lambda x: x.abs().max(dim=-1).values,
    "minimum": lambda x: x.min(dim=-1).values,
    "delta": lambda x: (x.max(dim=-1).values - x.mean(dim=-1)).abs(),
    "log_delta": lambda x: torch.log((x.max(dim=-1).values - x.mean(dim=-1)).abs() + 1e-10),
}


class FunctionKernelFeatureExtractor(FeatureExtractor):
    """Configurable selection that builds one fused kernel out of the requested features.

    `_compile` (config compile, unrelated to torch.compile) groups the requested
    `features`/`features_per_state` the same way ModuleNNFeatureExtractor does: per
    distinct feature name, the list of state indices that actually asked for it.
    Each feature's function is then closed over its own `index_select` (skipped
    when every state wants it, reusing the caller's tensor instead of paying for a
    copy - same free-view trick as ModuleNNFeatureExtractor's `_select_all_states`)
    so the reduction itself only ever touches the states that were actually
    requested - it does not compute a feature across every state and discard the
    unwanted ones afterward. `_kernel(y_bst)` runs every (func, states) pair and
    `torch.cat`s the results directly in the column order `feature_names`
    describes - no separate gather step, since each piece is already exactly the
    slice that belongs at that position. The whole composed call is optionally
    wrapped with `torch.compile` (`use_compile`, default True) as a single traced
    graph - no fallback: if `torch.compile` can't handle this config it raises, so
    the failure is visible instead of silently paying for both a compiled and an
    eager path. Reusing an instance for a different (num_states, device) than it
    was first compiled for is also an error - construct a new extractor instead of
    silently paying a hidden recompile.

    This trades eager-mode speed for a fair closures-vs-nn.Module comparison and
    for the best compiled number: `index_select` on the state axis (not the last
    dim) is a strided gather that has to materialize a copy of the whole (B, S, T)
    input, ~7x slower eager than the equivalent "reduce over every state, then
    gather the wanted cells from the small output" approach this class used
    before - torch.compile fuses the slice into the reduction and hides that cost
    entirely, which is why `use_compile=True` (the default here and the intended
    real-world configuration) is unaffected.
    """

    def __init__(
        self,
        features: Mapping[str, Any] | None = DEFAULT_TORCH_FC_PARAMETERS,
        features_per_state: dict[int, Mapping[str, Any] | None] | None = None,
        use_compile: bool = True,
    ):
        super().__init__(time_steady=0.0)
        self.features = features
        self.features_per_state = features_per_state or {}
        self.use_compile = use_compile
        self._kernel: Callable[[torch.Tensor], torch.Tensor] | None = None
        self._compiled_num_states: int | None = None
        self._compiled_device: torch.device | None = None
        self._feature_names: list[str] | None = None

    def _compile(self, num_states: int, device: torch.device) -> None:
        # Group by feature -> [states that want it], preserving the order features
        # are first requested - the same grouping ModuleNNFeatureExtractor uses, so
        # each feature's reduction only ever runs over the states that asked for it.
        per_feature_states: dict[str, list[int]] = {}
        for state_idx in range(num_states):
            state_features = self.features_per_state.get(state_idx, self.features)
            if state_features is None:
                continue
            for feature_name, parameters in state_features.items():
                if feature_name not in FEATURE_FUNCS_BST or parameters is not None:
                    raise ValueError(
                        "Only parameter-free features from DEFAULT_FEATURE_ORDER "
                        f"are supported, got {feature_name!r}: {parameters!r}"
                    )
                per_feature_states.setdefault(feature_name, []).append(state_idx)

        feature_names: list[str] = []
        # Each entry is (func, state_index tensor or None for "every state, reuse
        # the tensor as-is").
        slices: list[tuple[Callable[[torch.Tensor], torch.Tensor], torch.Tensor | None]] = []
        for feature_name, state_indices in per_feature_states.items():
            func = FEATURE_FUNCS_BST[feature_name]
            select_all_states = state_indices == list(range(num_states))
            state_index = (
                None
                if select_all_states
                else torch.tensor(state_indices, dtype=torch.long, device=device)
            )
            slices.append((func, state_index))
            feature_names.extend(
                format_feature_name(feature_name, state_index=state_idx)
                for state_idx in state_indices
            )

        def kernel_eager(
            y_bst: torch.Tensor,
            slices: tuple[
                tuple[Callable[[torch.Tensor], torch.Tensor], torch.Tensor | None], ...
            ] = tuple(slices),
        ) -> torch.Tensor:
            return torch.cat(
                [
                    func(y_bst if state_index is None else y_bst.index_select(1, state_index))
                    for func, state_index in slices
                ],
                dim=1,
            )

        self._kernel = (
            torch.compile(kernel_eager, dynamic=True) if self.use_compile else kernel_eager
        )

        self._feature_names = feature_names
        self._compiled_num_states = num_states
        self._compiled_device = device

    @torch.no_grad()
    def extract_features(self, y_bst: torch.Tensor) -> torch.Tensor:
        if y_bst.ndim != 3 or not y_bst.is_contiguous() or y_bst.stride(-1) != 1:
            raise ValueError(
                "Expected physical BST tensor (batch, state, time) with contiguous time"
            )

        num_states = y_bst.shape[1]
        if self._compiled_num_states is None:
            self._compile(num_states, y_bst.device)
        elif self._compiled_num_states != num_states or self._compiled_device != y_bst.device:
            raise ValueError(
                "This extractor was compiled for "
                f"(num_states={self._compiled_num_states}, device={self._compiled_device}); "
                f"got (num_states={num_states}, device={y_bst.device}). Construct a new "
                "extractor instead of silently recompiling for a different shape/device."
            )

        assert self._kernel is not None
        features = self._kernel(
            y_bst
        )  # torch.compile failures raise straight through - no silent fallback

        return impute_extreme(features)

    @property
    def feature_names(self) -> list[str]:
        if self._feature_names is None:
            raise RuntimeError("Feature names not available. Call extract_features first.")
        return self._feature_names

In [ ]:
# Column-by-column parity check, independent of column ordering (needed since
# FunctionKernelFeatureExtractor is state-major but ModuleNNFeatureExtractor
# below is feature-major for mixed per-state configs). Reuses the same per-feature
# tolerance reasoning as assert_feature_matrices_close in cell 3, applied to
# eager-vs-torch.compile drift instead of TSB-vs-BST drift: median/maximum/
# absolute_maximum/minimum are pure selection ops and must be bit-identical.
# mean/delta (and log_delta, by inverting it back to delta first) get a
# magnitude-scaled tolerance instead of a flat one: float32 reduction-order
# differences scale with eps * n_time * |source|, and delta = |max - mean| can
# subtract two similar-magnitude numbers down to something tiny, so a fixed
# atol/rtol either rejects well-conditioned states with large delta or is far
# too tight for states where cancellation shrinks delta toward zero.
def assert_feature_columns_close(
    expected: torch.Tensor,
    candidate: torch.Tensor,
    feature_names_per_column: list[str],
    source: torch.Tensor,
    label: str,
) -> None:
    if expected.shape != candidate.shape:
        raise AssertionError(
            f"{label}: shape mismatch {tuple(expected.shape)} != {tuple(candidate.shape)}"
        )

    n_time = source.shape[-1]
    max_abs_source = torch.maximum(
        source.amax(dim=-1).abs(), source.amin(dim=-1).abs()
    ).cpu()  # (B, S)
    reduction_atol = (8.0 * torch.finfo(source.dtype).eps * n_time * max_abs_source).clamp_min(1e-6)

    failures: list[str] = []
    for col, full_name in enumerate(feature_names_per_column):
        state_str, feature_name = full_name.split("__", 1)
        state_idx = int(state_str.removeprefix("state_"))
        expected_col = expected[:, col]
        candidate_col = candidate[:, col]
        atol_col = reduction_atol[:, state_idx]

        if feature_name in {"median", "maximum", "absolute_maximum", "minimum"}:
            valid = torch.equal(expected_col, candidate_col)
        elif feature_name in {"mean", "delta"}:
            valid = bool(torch.all((expected_col - candidate_col).abs() <= atol_col))
        elif feature_name == "log_delta":
            expected_delta = torch.exp(expected_col) - 1e-10
            candidate_delta = torch.exp(candidate_col) - 1e-10
            valid = bool(torch.all((expected_delta - candidate_delta).abs() <= atol_col))
        else:
            valid = torch.allclose(expected_col, candidate_col, atol=1e-5, rtol=1e-4)

        if not valid:
            diff = (expected_col - candidate_col).abs().max().item()
            failures.append(f"{full_name}={diff:.3e}")

    if failures:
        raise AssertionError(f"{label}: feature parity failed: {', '.join(failures)}")


# Repeats the whole benchmark_callable measurement (its own warmup + REPEATS
# samples) n_trials independent times. Reports min_ms as the headline number,
# not median: for latency microbenchmarks, interference (OS scheduling, GC,
# thermal throttling - all very real under WSL2) can only ever add time to a
# sample, never subtract it, so the minimum across all samples is the closest
# available estimate of true steady-state cost. median_ms/max_ms are kept
# alongside purely for diagnosis - a wide min/max spread flags a noisy run
# instead of a real per-configuration regression.
def benchmark_stats(
    label: str, device: str, fn: Callable[[], Any], n_trials: int = 5
) -> dict[str, float]:
    trials = [benchmark_callable(f"{label}_trial{trial}", device, fn) for trial in range(n_trials)]
    return {
        "min_ms": float(min(t["min_ms"] for t in trials)),
        "median_ms": float(pd.Series([t["median_ms"] for t in trials]).median()),
        "max_ms": float(max(t["max_ms"] for t in trials)),
    }


# torch.compile's TorchDynamo frame-compile counter. If this changes between two
# snapshots taken around a "should already be warm" benchmark window, the graph
# recompiled mid-benchmark - the measured time includes one-off compile overhead,
# not steady-state execution, which would explain an outlier "compiled" number
# without it being a bug in the extractor itself.
def frame_compile_count() -> int:
    import torch._dynamo as dynamo

    return dynamo.utils.counters.get("frames", {}).get("total", 0)


# Compare configuration overhead directly with the existing fused + BST result.
benchmark_configurations: dict[str, tuple[Mapping[str, Any] | None, dict[int, Any]]] = {
    "all features on all states": (DEFAULT_TORCH_FC_PARAMETERS, {}),
    "state 1: maximum + minimum": (
        None,
        {1: {"maximum": None, "minimum": None}},
    ),
    "state 0: maximum; state 1: skipped": (
        DEFAULT_TORCH_FC_PARAMETERS,
        {0: {"maximum": None}, 1: None},
    ),
}

function_kernel_rows: list[dict[str, float | int | str]] = []
for device in ["cpu", "gpu"] if CUDA_AVAILABLE else ["cpu"]:
    y_bst = y_bst_gpu if device == "gpu" else y_bst_cpu
    assert y_bst is not None

    # TorchFeatureExtractor takes a TSB-layout Solution, not a raw BST tensor.
    # solution_tsb_cpu/y_tsb_cpu from the earlier cell were already deleted there
    # (this section only keeps y_bst_* alive) - rebuild a TSB Solution here so
    # TorchFeatureExtractor can actually be re-measured per configuration below,
    # instead of reusing one cached "all features" number for every row.
    # filter_time DOES read solution.time (start_idx = first index where
    # time >= time_steady); with time_steady=0.0 it resolves to start_idx=0 only
    # because the solver already only ever saves points from t_eval=(1670, 2000)
    # onward - the solver, not the extractor, is what guarantees "steady only"
    # here. A monotonic linspace over that same already-steady range is exactly
    # as safe as the real (irregular, adaptive-step) times: every value clears
    # time_steady=0.0, so filter_time keeps 100% of rows either way - the values
    # aren't ignored, they just can't cause any filtering in this configuration.
    # initial_condition genuinely is unused (grep confirms no other code path
    # reads it from a Solution passed to extract_features), so the first time
    # slice is reused as a validly-shaped stand-in instead of a separate tensor.
    y_tsb = y_bst.permute(2, 0, 1).contiguous()
    solution_tsb = Solution(
        y_tsb[0],
        torch.linspace(*BENCHMARK_T_EVAL, y_tsb.shape[0], device=y_bst.device),
        y_tsb,
    )

    fused_bst_reference_call = lambda y_bst=y_bst: extract_fused_default_bst_device_resident(y_bst)
    fused_bst_ms = float(
        benchmark_callable(
            "fused_bst_device_resident_reference",
            device,
            fused_bst_reference_call,
        )["median_ms"]
    )

    for configuration, (features, features_per_state) in benchmark_configurations.items():
        extractor = FunctionKernelFeatureExtractor(
            features=features,
            features_per_state=features_per_state,
        )
        call = lambda extractor=extractor, y_bst=y_bst: extractor.extract_features(y_bst)
        # use_compile=False sibling, same config, to isolate torch.compile's own
        # contribution instead of just reporting one (compiled) number.
        extractor_uncompiled = FunctionKernelFeatureExtractor(
            features=features,
            features_per_state=features_per_state,
            use_compile=False,
        )
        call_uncompiled = (
            lambda extractor=extractor_uncompiled, y_bst=y_bst: extractor.extract_features(y_bst)
        )

        candidate = call()
        candidate_uncompiled = call_uncompiled()
        full_fused = extract_fused_default_bst_device_resident(y_bst)
        # Look up by name, not position: FunctionKernelFeatureExtractor now
        # groups columns feature-major (like ModuleNNFeatureExtractor), not
        # state-major, since it slices to the requested states before reducing
        # instead of computing every feature over every state and discarding the
        # unwanted ones - so this can no longer assume a fixed state-major layout.
        full_fused_names = [
            format_feature_name(name, state_index=state_idx)
            for state_idx in range(y_bst.shape[1])
            for name in DEFAULT_FEATURE_ORDER
        ]
        name_to_col = {name: idx for idx, name in enumerate(full_fused_names)}
        expected_columns = [name_to_col[name] for name in extractor.feature_names]
        expected = full_fused[:, expected_columns]
        if candidate.device != y_bst.device:
            raise AssertionError(
                f"FunctionKernel extractor moved output to {candidate.device}; expected {y_bst.device}"
            )
        if full_fused.device != y_bst.device:
            raise AssertionError(
                f"Fused BST reference moved output to {full_fused.device}; expected {y_bst.device}"
            )
        assert_feature_columns_close(
            expected.detach().cpu(),
            candidate.detach().cpu(),
            extractor.feature_names,
            y_bst,
            f"{configuration} / {device} / compiled",
        )
        # extractor_uncompiled has the identical config, so it produces the same
        # feature_names/column layout - only the kernel (compiled vs eager) differs.
        assert_feature_columns_close(
            expected.detach().cpu(),
            candidate_uncompiled.detach().cpu(),
            extractor.feature_names,
            y_bst,
            f"{configuration} / {device} / uncompiled",
        )
        output_features = candidate.shape[1]
        del candidate, candidate_uncompiled, full_fused, expected
        if device == "gpu":
            release_gpu_memory()

        compile_count_before = frame_compile_count()
        function_kernel_stats = benchmark_stats(
            f"function_kernel_{configuration}",
            device,
            call,
        )
        function_kernel_ms = function_kernel_stats["min_ms"]
        function_kernel_recompiled_during_timing = frame_compile_count() != compile_count_before
        if function_kernel_recompiled_during_timing:
            print(
                f"WARNING: torch.compile recompiled during the timed benchmark for "
                f"{configuration!r} / {device} (FunctionKernelFeatureExtractor) - "
                "function_kernel_ms includes compile overhead, not just "
                "steady-state execution."
            )
        function_kernel_uncompiled_stats = benchmark_stats(
            f"function_kernel_uncompiled_{configuration}",
            device,
            call_uncompiled,
        )
        function_kernel_uncompiled_ms = function_kernel_uncompiled_stats["min_ms"]

        # Re-measure TorchFeatureExtractor for THIS configuration specifically -
        # same feature subset, same states - instead of reusing the fixed
        # "all 20 features" number from the earlier focused-benchmark cell for
        # every row (which made partial configs look unfairly slower than they
        # actually are relative to the old extractor).
        torch_extractor = PublicTorchFeatureExtractor(
            time_steady=0.0,
            features=features,
            features_per_state=features_per_state,
            normalize=NORMALIZE,
            device=device,
        )
        torch_call = lambda torch_extractor=torch_extractor, solution_tsb=solution_tsb: (
            torch_extractor.extract_features(solution_tsb)
        )
        torch_candidate = torch_call()
        if torch_candidate.shape[1] != output_features:
            raise AssertionError(
                f"TorchFeatureExtractor produced {torch_candidate.shape[1]} columns; "
                f"expected {output_features} to match {configuration!r}"
            )
        del torch_candidate
        if device == "gpu":
            release_gpu_memory()
        torch_feature_extractor_ms = float(
            benchmark_callable(
                f"torch_feature_extractor_{configuration}",
                device,
                torch_call,
            )["median_ms"]
        )
        del torch_call, torch_extractor
        if device == "gpu":
            release_gpu_memory()

        function_kernel_rows.append({
            "device": device,
            "configuration": configuration,
            "output_features": output_features,
            "torch_feature_extractor_ms": torch_feature_extractor_ms,
            "function_kernel_ms": function_kernel_ms,
            "function_kernel_max_ms": function_kernel_stats["max_ms"],
            "function_kernel_uncompiled_ms": function_kernel_uncompiled_ms,
            "function_kernel_uncompiled_max_ms": function_kernel_uncompiled_stats["max_ms"],
            "function_kernel_compile_speedup": function_kernel_uncompiled_ms / function_kernel_ms,
            "function_kernel_recompiled_during_timing": function_kernel_recompiled_during_timing,
            "torch_feature_extractor_over_function_kernel_speedup": torch_feature_extractor_ms
            / function_kernel_ms,
            "fused_bst_reference_ms": (
                fused_bst_ms if configuration == "all features on all states" else np.nan
            ),
            "function_kernel_time_over_fused_bst": (
                function_kernel_ms / fused_bst_ms
                if configuration == "all features on all states"
                else np.nan
            ),
        })
        del call, call_uncompiled, extractor, extractor_uncompiled
        if device == "gpu":
            release_gpu_memory()

    del y_tsb, solution_tsb
    if device == "gpu":
        release_gpu_memory()

function_kernel_df = pd.DataFrame(function_kernel_rows)
print("FunctionKernelFeatureExtractor versus the fused + BST reference:")
display(
    function_kernel_df[
        [
            "device",
            "configuration",
            "output_features",
            "torch_feature_extractor_ms",
            "function_kernel_ms",
            "function_kernel_max_ms",
            "function_kernel_uncompiled_ms",
            "function_kernel_uncompiled_max_ms",
            "function_kernel_compile_speedup",
            "function_kernel_recompiled_during_timing",
            "torch_feature_extractor_over_function_kernel_speedup",
            "fused_bst_reference_ms",
            "function_kernel_time_over_fused_bst",
        ]
    ].round(3)
)

for device in function_kernel_df["device"].unique():
    default_row = function_kernel_df[
        (function_kernel_df["device"] == device)
        & (function_kernel_df["configuration"] == "all features on all states")
    ].iloc[0]
    ratio = float(default_row["function_kernel_time_over_fused_bst"])
    torch_vs_function_kernel = float(
        default_row["torch_feature_extractor_over_function_kernel_speedup"]
    )
    compile_speedup = float(default_row["function_kernel_compile_speedup"])
    print(
        f"{device.upper()}: FunctionKernelFeatureExtractor is "
        f"{torch_vs_function_kernel:.2f}x faster than TorchFeatureExtractor, and takes "
        f"{ratio:.2f}x the fused + BST time. torch.compile itself is "
        f"{compile_speedup:.2f}x faster than the identical eager kernel."
    )


# y_bst_cpu / y_bst_gpu are still needed by the ModuleNNFeatureExtractor
# comparison below - cleanup moved to the end of that cell instead of here.

In [ ]:
# One nn.Module per feature, bound to which states to extract from at construction
# ("arg1, arg2" pre-bound as __init__ arguments, not curried functions) - so
# `forward(x)` is a plain, pure `(x) -> result` callable. A ModuleList holds all
# requested feature modules; the extractor's job is turning the user-friendly
# features/features_per_state config into that ModuleList plus a feature_names list
# built in lockstep, then optionally handing the whole thing to torch.compile.
# Kept side-by-side with FunctionKernelFeatureExtractor (not replacing it)
# so the two designs can be compared directly below.


class FeatureModule(nn.Module):
    """One feature, bound to its state indices at construction.

    `state_indices` is fixed as an __init__ argument, so forward(x) never needs to
    be told which states it applies to - that's already baked in.
    """

    def __init__(self, state_indices: list[int], num_states: int):
        super().__init__()
        self.register_buffer(
            "state_indices", torch.tensor(state_indices, dtype=torch.long), persistent=False
        )
        # Selecting every state, in order, is decided once at construction from
        # plain Python ints (not tensor data), so forward() can reuse the caller's
        # tensor directly instead of paying index_select's copy - the same
        # free-view trick proved for the reshape-vs-gather case above, now applied
        # per feature module instead of once globally.
        self._select_all_states = state_indices == list(range(num_states))

    def forward(self, y_bst: torch.Tensor) -> torch.Tensor:
        x_selected = y_bst if self._select_all_states else y_bst.index_select(1, self.state_indices)
        return self.reduce(x_selected)

    def reduce(self, x_selected: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class MedianFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.median(dim=-1).values


class MeanFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.mean(dim=-1)


class StandardDeviationFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.std(dim=-1, correction=0)


class VarianceFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.var(dim=-1, correction=0)


class RootMeanSquareFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sqrt((x**2).mean(dim=-1))


class MaximumFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.max(dim=-1).values


class AbsoluteMaximumFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.abs().max(dim=-1).values


class MinimumFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return x.min(dim=-1).values


class DeltaFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        return (x.max(dim=-1).values - x.mean(dim=-1)).abs()


class LogDeltaFeature(FeatureModule):
    def reduce(self, x: torch.Tensor) -> torch.Tensor:
        delta_x = (x.max(dim=-1).values - x.mean(dim=-1)).abs()
        return torch.log(delta_x + 1e-10)


FEATURE_MODULE_CLASSES: dict[str, type[FeatureModule]] = {
    "median": MedianFeature,
    "mean": MeanFeature,
    "standard_deviation": StandardDeviationFeature,
    "variance": VarianceFeature,
    "root_mean_square": RootMeanSquareFeature,
    "maximum": MaximumFeature,
    "absolute_maximum": AbsoluteMaximumFeature,
    "minimum": MinimumFeature,
    "delta": DeltaFeature,
    "log_delta": LogDeltaFeature,
}


class FusedFeatureModuleList(nn.Module):
    """Concatenates every feature module's (B, K_i) output along dim=-1, in module order."""

    def __init__(self, feature_modules: list[FeatureModule]):
        super().__init__()
        self.feature_modules = nn.ModuleList(feature_modules)

    def forward(self, y_bst: torch.Tensor) -> torch.Tensor:
        return torch.cat([module(y_bst) for module in self.feature_modules], dim=-1)


class ModuleNNFeatureExtractor(FeatureExtractor):
    """Same configuration surface as FunctionKernelFeatureExtractor, built from
    one nn.Module per requested feature instead of a fused index/gather kernel.

    `_compile` (config compile, unrelated to torch.compile) reorganizes the
    state-indexed `features`/`features_per_state` config into (feature -> [state
    indices that want it]), builds one nn.Module per distinct feature already bound
    to its own state indices, and builds `feature_names` in lockstep so column i of
    the concatenated ModuleList output is named `feature_names[i]` - even when one
    feature module covers 3 states and another covers 2. The whole ModuleList is
    then optionally wrapped with `torch.compile` (`use_compile`, default True) as a
    single traced call - no fallback: if `torch.compile` can't handle this config it
    raises, so the failure is visible instead of silently paying for both a compiled
    and an eager path. Reusing an instance for a different (num_states, device) than
    it was first compiled for is also an error - construct a new extractor instead
    of silently paying a hidden recompile.
    """

    def __init__(
        self,
        features: Mapping[str, Any] | None = DEFAULT_TORCH_FC_PARAMETERS,
        features_per_state: dict[int, Mapping[str, Any] | None] | None = None,
        use_compile: bool = True,
    ):
        super().__init__(time_steady=0.0)
        self.features = features
        self.features_per_state = features_per_state or {}
        self.use_compile = use_compile
        self._module_list: FusedFeatureModuleList | None = None
        self._kernel: Callable[[torch.Tensor], torch.Tensor] | None = None
        self._compiled_num_states: int | None = None
        self._compiled_device: torch.device | None = None
        self._extracted_feature_names: list[str] | None = None

    def _compile(self, num_states: int, device: torch.device) -> None:
        # Reorganize the (state-indexed) user config into (feature -> [state
        # indices that want it]), preserving the order features are first requested.
        per_feature_states: dict[str, list[int]] = {}
        for state_idx in range(num_states):
            state_features = self.features_per_state.get(state_idx, self.features)
            if state_features is None:
                continue
            for feature_name, parameters in state_features.items():
                if feature_name not in FEATURE_MODULE_CLASSES or parameters is not None:
                    raise ValueError(
                        "Only parameter-free features from DEFAULT_FEATURE_ORDER "
                        f"are supported, got {feature_name!r}: {parameters!r}"
                    )
                per_feature_states.setdefault(feature_name, []).append(state_idx)

        feature_modules: list[FeatureModule] = []
        feature_names: list[str] = []
        for feature_name, state_indices in per_feature_states.items():
            module_cls = FEATURE_MODULE_CLASSES[feature_name]
            feature_modules.append(module_cls(state_indices, num_states))
            feature_names.extend(
                format_feature_name(feature_name, state_index=state_idx)
                for state_idx in state_indices
            )

        fused = FusedFeatureModuleList(feature_modules).to(device)
        fused.eval()

        self._module_list = fused
        self._kernel = torch.compile(fused, dynamic=True) if self.use_compile else fused
        self._extracted_feature_names = feature_names
        self._compiled_num_states = num_states
        self._compiled_device = device

    @torch.no_grad()
    def extract_features(self, y_bst: torch.Tensor) -> torch.Tensor:
        if y_bst.ndim != 3 or not y_bst.is_contiguous() or y_bst.stride(-1) != 1:
            raise ValueError(
                "Expected physical BST tensor (batch, state, time) with contiguous time"
            )

        num_states = y_bst.shape[1]
        if self._compiled_num_states is None:
            self._compile(num_states, y_bst.device)
        elif self._compiled_num_states != num_states or self._compiled_device != y_bst.device:
            raise ValueError(
                "This extractor was compiled for "
                f"(num_states={self._compiled_num_states}, device={self._compiled_device}); "
                f"got (num_states={num_states}, device={y_bst.device}). Construct a new "
                "extractor instead of silently recompiling for a different shape/device."
            )

        assert self._kernel is not None
        features = self._kernel(
            y_bst
        )  # torch.compile failures raise straight through - no silent fallback

        return impute_extreme(features)

    @property
    def feature_names(self) -> list[str]:
        if self._extracted_feature_names is None:
            raise RuntimeError("Feature names not available. Call extract_features first.")
        return self._extracted_feature_names

In [ ]:
# Compare ModuleNNFeatureExtractor against FunctionKernelFeatureExtractor on
# the same configurations, validated against the same fused + BST reference.
# Column order can differ between the two (state-major for the function-kernel/gather
# extractor, feature-major for the module-nn extractor when configs mix per-state
# feature sets) - validate by name, not by position, using each extractor's own
# feature_names.
module_nn_rows: list[dict[str, float | int | str]] = []
for device in ["cpu", "gpu"] if CUDA_AVAILABLE else ["cpu"]:
    y_bst = y_bst_gpu if device == "gpu" else y_bst_cpu
    assert y_bst is not None
    full_fused = extract_fused_default_bst_device_resident(y_bst)
    full_fused_names = [
        format_feature_name(name, state_index=state_idx)
        for state_idx in range(y_bst.shape[1])
        for name in DEFAULT_FEATURE_ORDER
    ]
    name_to_col = {name: idx for idx, name in enumerate(full_fused_names)}

    for configuration, (features, features_per_state) in benchmark_configurations.items():
        extractor = ModuleNNFeatureExtractor(
            features=features,
            features_per_state=features_per_state,
        )
        call = lambda extractor=extractor, y_bst=y_bst: extractor.extract_features(y_bst)
        # use_compile=False sibling, same config, to isolate torch.compile's own
        # contribution instead of just reporting one (compiled) number.
        extractor_uncompiled = ModuleNNFeatureExtractor(
            features=features,
            features_per_state=features_per_state,
            use_compile=False,
        )
        call_uncompiled = (
            lambda extractor=extractor_uncompiled, y_bst=y_bst: extractor.extract_features(y_bst)
        )

        candidate = call()
        candidate_uncompiled = call_uncompiled()
        expected_columns = [name_to_col[name] for name in extractor.feature_names]
        expected = full_fused[:, expected_columns]
        if candidate.device != y_bst.device:
            raise AssertionError(
                f"ModuleNNFeatureExtractor moved output to {candidate.device}; expected {y_bst.device}"
            )
        assert_feature_columns_close(
            expected.detach().cpu(),
            candidate.detach().cpu(),
            extractor.feature_names,
            y_bst,
            f"{configuration} / {device} / compiled",
        )
        # extractor_uncompiled has the identical config, so it produces the same
        # feature_names/column layout - only the kernel (compiled vs eager) differs.
        assert_feature_columns_close(
            expected.detach().cpu(),
            candidate_uncompiled.detach().cpu(),
            extractor.feature_names,
            y_bst,
            f"{configuration} / {device} / uncompiled",
        )
        output_features = candidate.shape[1]
        del candidate, candidate_uncompiled, expected
        if device == "gpu":
            release_gpu_memory()

        compile_count_before = frame_compile_count()
        module_nn_stats = benchmark_stats(
            f"module_nn_{configuration}",
            device,
            call,
        )
        module_nn_ms = module_nn_stats["min_ms"]
        module_nn_recompiled_during_timing = frame_compile_count() != compile_count_before
        if module_nn_recompiled_during_timing:
            print(
                f"WARNING: torch.compile recompiled during the timed benchmark for "
                f"{configuration!r} / {device} (ModuleNNFeatureExtractor) - "
                "module_nn_ms includes compile overhead, not just "
                "steady-state execution."
            )
        module_nn_uncompiled_stats = benchmark_stats(
            f"module_nn_uncompiled_{configuration}",
            device,
            call_uncompiled,
        )
        module_nn_uncompiled_ms = module_nn_uncompiled_stats["min_ms"]

        function_kernel_row = function_kernel_df[
            (function_kernel_df["device"] == device)
            & (function_kernel_df["configuration"] == configuration)
        ].iloc[0]
        function_kernel_ms = float(function_kernel_row["function_kernel_ms"])
        torch_feature_extractor_ms = float(function_kernel_row["torch_feature_extractor_ms"])

        module_nn_rows.append({
            "device": device,
            "configuration": configuration,
            "output_features": output_features,
            "function_kernel_ms": function_kernel_ms,
            "module_nn_ms": module_nn_ms,
            "module_nn_max_ms": module_nn_stats["max_ms"],
            "module_nn_uncompiled_ms": module_nn_uncompiled_ms,
            "module_nn_uncompiled_max_ms": module_nn_uncompiled_stats["max_ms"],
            "module_nn_compile_speedup": module_nn_uncompiled_ms / module_nn_ms,
            "module_nn_recompiled_during_timing": module_nn_recompiled_during_timing,
            # >1 means ModuleNN is that many times faster than FunctionKernel (FunctionKernel_ms / ModuleNN_ms),
            # matching the "X_over_Y_speedup = X_ms / Y_ms" convention used above for
            # torch_feature_extractor_over_function_kernel_speedup.
            "function_kernel_over_module_nn_speedup": function_kernel_ms / module_nn_ms,
            "torch_feature_extractor_ms": torch_feature_extractor_ms,
            "torch_feature_extractor_over_module_nn_speedup": torch_feature_extractor_ms
            / module_nn_ms,
        })
        del call, call_uncompiled, extractor, extractor_uncompiled
        if device == "gpu":
            release_gpu_memory()
    del full_fused

module_nn_df = pd.DataFrame(module_nn_rows)
print("ModuleNNFeatureExtractor versus FunctionKernelFeatureExtractor:")
display(
    module_nn_df[
        [
            "device",
            "configuration",
            "output_features",
            "torch_feature_extractor_ms",
            "function_kernel_ms",
            "module_nn_ms",
            "module_nn_max_ms",
            "module_nn_uncompiled_ms",
            "module_nn_uncompiled_max_ms",
            "module_nn_compile_speedup",
            "module_nn_recompiled_during_timing",
            "function_kernel_over_module_nn_speedup",
            "torch_feature_extractor_over_module_nn_speedup",
        ]
    ].round(3)
)

for _, row in module_nn_df.iterrows():
    relation = "faster" if row["function_kernel_over_module_nn_speedup"] > 1 else "slower"
    print(
        f"{row['device'].upper()} / {row['configuration']}: ModuleNNFeatureExtractor is "
        f"{max(row['function_kernel_over_module_nn_speedup'], 1 / row['function_kernel_over_module_nn_speedup']):.2f}x "
        f"{relation} than FunctionKernelFeatureExtractor "
        f"({row['module_nn_ms']:.3f} ms vs {row['function_kernel_ms']:.3f} ms), "
        f"and {row['torch_feature_extractor_over_module_nn_speedup']:.2f}x faster than "
        f"TorchFeatureExtractor ({row['module_nn_ms']:.3f} ms vs "
        f"{row['torch_feature_extractor_ms']:.3f} ms). torch.compile itself gives "
        f"ModuleNNFeatureExtractor a {row['module_nn_compile_speedup']:.2f}x speedup over its own "
        "eager kernel."
    )

# y_bst_cpu / y_bst_gpu are still needed by the compile-timing cell below -
# cleanup moved to the end of that cell instead of here.

In [ ]:
# Bare-class-per-feature variant: no nn.Module, no ModuleList - testing whether
# nn.Module itself (not the underlying math, which is identical across all three
# designs here) is what caused ModuleNNFeatureExtractor's CPU regression above.
# __init__(params) bakes in state selection (built once, on-device, exactly like
# FunctionKernelFeatureExtractor's `slices`), __call__(x) is the pure
# kernel - no hooks, no training-mode branch, no buffer-registration machinery.
# Reuses FEATURE_FUNCS_BST so the only variable under test is the container
# mechanism (closures vs nn.Module vs bare class), not the math.


class BaseFeature:
    """One feature, bound to its state indices at construction - the bare-class
    counterpart to FeatureModule (cell above), no nn.Module inheritance."""

    def __init__(
        self,
        state_indices: list[int],
        num_states: int,
        reduce_fn: Callable[[torch.Tensor], torch.Tensor],
        device: torch.device,
    ):
        select_all_states = state_indices == list(range(num_states))
        self.state_index = (
            None
            if select_all_states
            else torch.tensor(state_indices, dtype=torch.long, device=device)
        )
        self._reduce_fn = reduce_fn

    def __call__(self, y_bst: torch.Tensor) -> torch.Tensor:
        x_selected = y_bst if self.state_index is None else y_bst.index_select(1, self.state_index)
        return self._reduce_fn(x_selected)


class FusedBaseFeatures:
    """Concatenates every bare feature's (B, K_i) output along dim=-1, in order -
    the bare-class counterpart to FusedFeatureModuleList."""

    def __init__(self, features: list[BaseFeature]):
        self.features = features

    def __call__(self, y_bst: torch.Tensor) -> torch.Tensor:
        return torch.cat([f(y_bst) for f in self.features], dim=-1)


class BaseClassFeatureExtractor(FeatureExtractor):
    """Same configuration surface as FunctionKernelFeatureExtractor and
    ModuleNNFeatureExtractor, built from one bare (non-nn.Module) class instance per
    requested feature instead of a fused index/gather kernel or an nn.Module.
    """

    def __init__(
        self,
        features: Mapping[str, Any] | None = DEFAULT_TORCH_FC_PARAMETERS,
        features_per_state: dict[int, Mapping[str, Any] | None] | None = None,
        use_compile: bool = True,
    ):
        super().__init__(time_steady=0.0)
        self.features = features
        self.features_per_state = features_per_state or {}
        self.use_compile = use_compile
        self._kernel: Callable[[torch.Tensor], torch.Tensor] | None = None
        self._compiled_num_states: int | None = None
        self._compiled_device: torch.device | None = None
        self._feature_names: list[str] | None = None

    def _compile(self, num_states: int, device: torch.device) -> None:
        per_feature_states: dict[str, list[int]] = {}
        for state_idx in range(num_states):
            state_features = self.features_per_state.get(state_idx, self.features)
            if state_features is None:
                continue
            for feature_name, parameters in state_features.items():
                if feature_name not in FEATURE_FUNCS_BST or parameters is not None:
                    raise ValueError(
                        "Only parameter-free features from DEFAULT_FEATURE_ORDER "
                        f"are supported, got {feature_name!r}: {parameters!r}"
                    )
                per_feature_states.setdefault(feature_name, []).append(state_idx)

        bare_features: list[BaseFeature] = []
        feature_names: list[str] = []
        for feature_name, state_indices in per_feature_states.items():
            bare_features.append(
                BaseFeature(state_indices, num_states, FEATURE_FUNCS_BST[feature_name], device)
            )
            feature_names.extend(
                format_feature_name(feature_name, state_index=state_idx)
                for state_idx in state_indices
            )

        fused = FusedBaseFeatures(bare_features)

        self._kernel = torch.compile(fused, dynamic=True) if self.use_compile else fused
        self._feature_names = feature_names
        self._compiled_num_states = num_states
        self._compiled_device = device

    @torch.no_grad()
    def extract_features(self, y_bst: torch.Tensor) -> torch.Tensor:
        if y_bst.ndim != 3 or not y_bst.is_contiguous() or y_bst.stride(-1) != 1:
            raise ValueError(
                "Expected physical BST tensor (batch, state, time) with contiguous time"
            )

        num_states = y_bst.shape[1]
        if self._compiled_num_states is None:
            self._compile(num_states, y_bst.device)
        elif self._compiled_num_states != num_states or self._compiled_device != y_bst.device:
            raise ValueError(
                "This extractor was compiled for "
                f"(num_states={self._compiled_num_states}, device={self._compiled_device}); "
                f"got (num_states={num_states}, device={y_bst.device}). Construct a new "
                "extractor instead of silently recompiling for a different shape/device."
            )

        assert self._kernel is not None
        features = self._kernel(
            y_bst
        )  # torch.compile failures raise straight through - no silent fallback

        return impute_extreme(features)

    @property
    def feature_names(self) -> list[str]:
        if self._feature_names is None:
            raise RuntimeError("Feature names not available. Call extract_features first.")
        return self._feature_names

In [ ]:
# Compare BaseClassFeatureExtractor (no nn.Module, no ModuleList) against
# FunctionKernelFeatureExtractor and ModuleNNFeatureExtractor on the same
# configurations - isolates whether nn.Module itself (not the underlying math,
# identical across all three designs) is what caused ModuleNNFeatureExtractor's
# CPU regression measured above.
base_class_rows: list[dict[str, float | int | str]] = []
for device in ["cpu", "gpu"] if CUDA_AVAILABLE else ["cpu"]:
    y_bst = y_bst_gpu if device == "gpu" else y_bst_cpu
    assert y_bst is not None
    full_fused = extract_fused_default_bst_device_resident(y_bst)
    full_fused_names = [
        format_feature_name(name, state_index=state_idx)
        for state_idx in range(y_bst.shape[1])
        for name in DEFAULT_FEATURE_ORDER
    ]
    name_to_col = {name: idx for idx, name in enumerate(full_fused_names)}

    for configuration, (features, features_per_state) in benchmark_configurations.items():
        extractor = BaseClassFeatureExtractor(
            features=features,
            features_per_state=features_per_state,
        )
        call = lambda extractor=extractor, y_bst=y_bst: extractor.extract_features(y_bst)
        extractor_uncompiled = BaseClassFeatureExtractor(
            features=features,
            features_per_state=features_per_state,
            use_compile=False,
        )
        call_uncompiled = (
            lambda extractor=extractor_uncompiled, y_bst=y_bst: extractor.extract_features(y_bst)
        )

        candidate = call()
        candidate_uncompiled = call_uncompiled()
        expected_columns = [name_to_col[name] for name in extractor.feature_names]
        expected = full_fused[:, expected_columns]
        if candidate.device != y_bst.device:
            raise AssertionError(
                f"BaseClassFeatureExtractor moved output to {candidate.device}; expected {y_bst.device}"
            )
        assert_feature_columns_close(
            expected.detach().cpu(),
            candidate.detach().cpu(),
            extractor.feature_names,
            y_bst,
            f"{configuration} / {device} / compiled",
        )
        assert_feature_columns_close(
            expected.detach().cpu(),
            candidate_uncompiled.detach().cpu(),
            extractor.feature_names,
            y_bst,
            f"{configuration} / {device} / uncompiled",
        )
        output_features = candidate.shape[1]
        del candidate, candidate_uncompiled, expected
        if device == "gpu":
            release_gpu_memory()

        compile_count_before = frame_compile_count()
        bare_stats = benchmark_stats(
            f"base_class_{configuration}",
            device,
            call,
        )
        bare_ms = bare_stats["min_ms"]
        bare_recompiled_during_timing = frame_compile_count() != compile_count_before
        if bare_recompiled_during_timing:
            print(
                f"WARNING: torch.compile recompiled during the timed benchmark for "
                f"{configuration!r} / {device} (BaseClassFeatureExtractor) - "
                "base_class_ms includes compile overhead, not just "
                "steady-state execution."
            )
        bare_uncompiled_stats = benchmark_stats(
            f"base_class_uncompiled_{configuration}",
            device,
            call_uncompiled,
        )
        bare_uncompiled_ms = bare_uncompiled_stats["min_ms"]

        function_kernel_row = function_kernel_df[
            (function_kernel_df["device"] == device)
            & (function_kernel_df["configuration"] == configuration)
        ].iloc[0]
        function_kernel_ms = float(function_kernel_row["function_kernel_ms"])

        module_nn_row = module_nn_df[
            (module_nn_df["device"] == device) & (module_nn_df["configuration"] == configuration)
        ].iloc[0]
        module_nn_ms = float(module_nn_row["module_nn_ms"])

        base_class_rows.append({
            "device": device,
            "configuration": configuration,
            "output_features": output_features,
            "function_kernel_ms": function_kernel_ms,
            "module_nn_ms": module_nn_ms,
            "base_class_ms": bare_ms,
            "base_class_max_ms": bare_stats["max_ms"],
            "base_class_uncompiled_ms": bare_uncompiled_ms,
            "base_class_uncompiled_max_ms": bare_uncompiled_stats["max_ms"],
            "base_class_compile_speedup": bare_uncompiled_ms / bare_ms,
            "base_class_recompiled_during_timing": bare_recompiled_during_timing,
            "function_kernel_over_base_class_speedup": function_kernel_ms / bare_ms,
            "module_nn_over_base_class_speedup": module_nn_ms / bare_ms,
        })
        del call, call_uncompiled, extractor, extractor_uncompiled
        if device == "gpu":
            release_gpu_memory()
    del full_fused

base_class_df = pd.DataFrame(base_class_rows)
print(
    "BaseClassFeatureExtractor versus FunctionKernelFeatureExtractor and ModuleNNFeatureExtractor:"
)
display(
    base_class_df[
        [
            "device",
            "configuration",
            "output_features",
            "function_kernel_ms",
            "module_nn_ms",
            "base_class_ms",
            "base_class_max_ms",
            "base_class_uncompiled_ms",
            "base_class_compile_speedup",
            "base_class_recompiled_during_timing",
            "function_kernel_over_base_class_speedup",
            "module_nn_over_base_class_speedup",
        ]
    ].round(3)
)

for _, row in base_class_df.iterrows():
    relation = "faster" if row["function_kernel_over_base_class_speedup"] > 1 else "slower"
    module_nn_relation = "faster" if row["module_nn_over_base_class_speedup"] > 1 else "slower"
    print(
        f"{row['device'].upper()} / {row['configuration']}: BaseClassFeatureExtractor is "
        f"{max(row['function_kernel_over_base_class_speedup'], 1 / row['function_kernel_over_base_class_speedup']):.2f}x "
        f"{relation} than FunctionKernelFeatureExtractor "
        f"({row['base_class_ms']:.3f} ms vs {row['function_kernel_ms']:.3f} ms), "
        f"and {max(row['module_nn_over_base_class_speedup'], 1 / row['module_nn_over_base_class_speedup']):.2f}x "
        f"{module_nn_relation} than ModuleNNFeatureExtractor "
        f"({row['base_class_ms']:.3f} ms vs {row['module_nn_ms']:.3f} ms)."
    )

## torch.compile warmup cost

Every benchmark above deliberately excludes `torch.compile`'s one-time tracing
and codegen cost: each extractor's first call (used only for the correctness
check) absorbs it before `median_of_trials` starts timing. This section
measures that excluded cost directly - first call (compile + execute) versus
second call (already compiled) - for a *fresh* instance of each extractor, per
configuration and device.


In [ ]:
def time_first_and_second_call(
    device: str,
    build_extractor: Callable[[], Any],
    input_tensor: torch.Tensor,
) -> tuple[float, float]:
    """First call = torch.compile tracing/codegen + execution; second call =
    already-compiled steady-state execution, for a brand-new extractor instance."""
    extractor = build_extractor()

    synchronize(device)
    t0 = perf_counter()
    extractor.extract_features(input_tensor)
    synchronize(device)
    first_call_ms = (perf_counter() - t0) * 1000.0

    t0 = perf_counter()
    extractor.extract_features(input_tensor)
    synchronize(device)
    second_call_ms = (perf_counter() - t0) * 1000.0

    del extractor
    if device == "gpu":
        release_gpu_memory()
    return first_call_ms, second_call_ms


compile_timing_rows: list[dict[str, float | int | str]] = []
for device in ["cpu", "gpu"] if CUDA_AVAILABLE else ["cpu"]:
    y_bst = y_bst_gpu if device == "gpu" else y_bst_cpu
    assert y_bst is not None

    for configuration, (features, features_per_state) in benchmark_configurations.items():
        for extractor_name, extractor_cls in (
            ("FunctionKernelFeatureExtractor", FunctionKernelFeatureExtractor),
            ("ModuleNNFeatureExtractor", ModuleNNFeatureExtractor),
            ("BaseClassFeatureExtractor", BaseClassFeatureExtractor),
        ):
            first_ms, second_ms = time_first_and_second_call(
                device,
                lambda features=features,
                features_per_state=features_per_state,
                extractor_cls=extractor_cls: (
                    extractor_cls(features=features, features_per_state=features_per_state)
                ),
                y_bst,
            )
            compile_timing_rows.append({
                "device": device,
                "configuration": configuration,
                "extractor": extractor_name,
                "first_call_ms": first_ms,
                "second_call_ms": second_ms,
                "compile_overhead_ms": first_ms - second_ms,
            })

compile_timing_df = pd.DataFrame(compile_timing_rows)
print(
    "torch.compile one-time cost: first call (compile + execute) versus second call (already compiled):"
)
display(compile_timing_df.round(3))

for _, row in compile_timing_df.iterrows():
    print(
        f"{row['device'].upper()} / {row['configuration']} / {row['extractor']}: "
        f"first call {row['first_call_ms']:.1f} ms, second call {row['second_call_ms']:.3f} ms "
        f"-> ~{row['compile_overhead_ms']:.1f} ms of one-time compile overhead."
    )

del y_bst_cpu, y_bst_gpu
release_gpu_memory()